## Mapping Barcodes and Cleaning Data

In [ ]:
# Original from 
# paulssonlab/src/paulssonlab/
# deaton/trenchripper/dev_notebooks/2021-11-10_lDE15_Final_1_Combined_Analysis_v1

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
from dask.distributed import wait
from matplotlib.ticker import FuncFormatter

import scipy.stats
from sklearn.linear_model import LinearRegression

dask_wd = "/home/lag36/scratch/lag36/dask"

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="04:00:00",
    local=False,
    n_workers=60,#100,
    n_workers_min=15,#100,
    memory="32GB",#"16GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

#### Import Barcode Dataframe

In [ ]:
# Read the file: /home/lag36/scratch/lag36/2023-10-31_lLAG2_AF555-AllCycles/Barcodes/metadata.hdf5
# using pandas
bc_df = pd.read_hdf(
    # "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/barcode_output_df_.hdf5",
    "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/barcode_output_df_thresh_0.hdf5",
    read_metadata=True,
)

In [ ]:
# meta_handle = tr.pandas_hdf5_handler("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/barcode_output_df.hdf5")
meta_handle = tr.pandas_hdf5_handler("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/barcode_output_df_thresh_0.hdf5")
pandas_barcode_df = meta_handle.read_df("barcodes",read_metadata=True)
barcode_df = dd.from_pandas(pandas_barcode_df,npartitions=1,sort=True)
# barcode_df = dd.from_pandas(pandas_barcode_df,npartitions=500,sort=True)
barcode_df = barcode_df.persist()

ttl_called = len(barcode_df.index)
ttl_trenches = pandas_barcode_df.metadata['Total Trenches']
ttl_trenches_w_cells = pandas_barcode_df.metadata['Total Trenches With Cells']
percent_called = ttl_called/ttl_trenches
percent_called_w_cells = ttl_called/ttl_trenches_w_cells

print(ttl_called)
print(ttl_trenches)
print(ttl_trenches_w_cells)
print(percent_called)
print(percent_called_w_cells)

### Import Analysis 

In [ ]:
analysis_df = dd.read_parquet("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Growth_Division/analysis",calculate_divisions=True)
last_trenchid = int(analysis_df.tail(1)["trenchid"])

In [ ]:
import skimage as sk
def hrm_find_mode(series, max_iter=1000, min_binsize=50):
    working_series = series
    for i in range(max_iter):
        range_max,range_min = np.max(working_series),np.min(working_series)
        midpoint = (range_max+range_min)/2
        above_middle = working_series[working_series>midpoint]
        below_middle = working_series[working_series<=midpoint]
        
        count_above = len(above_middle)
        count_below = len(below_middle)
        
        
        if count_above>count_below:
            working_series = above_middle
        else:
            working_series = below_middle
        
        if i > 0:
            if (len(working_series)<min_binsize) or (last_midpoint==midpoint):
                return np.mean(working_series)

        last_midpoint = midpoint
        
def bootstrap_hrm(series, n_bootstraps=100, max_n_per_bootstrap=100):
    modes = []
    
    series_len = len(series)
    
    n_per_bootstrap = min(series_len,max_n_per_bootstrap)

    for n in range(n_bootstraps):
        modes.append(hrm_find_mode(series.sample(n=n_per_bootstrap)))
    return np.mean(modes)
    
def get_GFPpos_modes(GFP_series,series_groupby,frac=0.01,n_bootstraps=100,max_n_per_bootstrap=100):
    gfp_vals = GFP_series.sample(frac=frac).compute().values
    tri_thr = sk.filters.threshold_triangle(gfp_vals)
    mode_series = series_groupby.apply(lambda x: bootstrap_hrm(x[x>tri_thr],n_bootstraps=n_bootstraps,max_n_per_bootstrap=max_n_per_bootstrap)).compute().sort_index()
    return mode_series

### Variables over FOV

In [ ]:
correct_fov = False ## turning this off, since the fluorescent data are internally normalized by the ratio measurement
values_to_rescale = ['mCherry-Penta mean_intensity', 'GFP-Penta mean_intensity']

analysis_df_nobkd = analysis_df[analysis_df["Objectid"]!=0]

In [ ]:
fig = plt.figure(figsize=(30,20))
values_names = ['Median mCherry Intensity', 'Median GFPmut2 Intensity']

for i,label in enumerate(values_to_rescale):
    fov_series_groupby = analysis_df_nobkd.groupby("fov")[label]
    if label == 'mCherry-Penta mean_intensity':
        fov_mode_series = fov_series_groupby.apply(lambda x: bootstrap_hrm(x,max_n_per_bootstrap=100)).compute().sort_index()
    elif label == 'GFP-Penta mean_intensity':
        fov_mode_series = get_GFPpos_modes(analysis_df['GFP-Penta mean_intensity'],fov_series_groupby,max_n_per_bootstrap=100)
    else:
        print("Weird Label")
    
    fov_correction_series = fov_mode_series/np.max(fov_mode_series)
    fov_correction_dict = fov_correction_series.to_dict()
    plt.subplot(2,3,i+1)
    plt.plot(fov_correction_series)
    plt.title(values_names[i],fontsize=22)
    plt.xlabel("FOV #", fontsize=18)
    plt.ylabel("Scaling", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.ylim(0.,1.)
    if correct_fov:
        label_scaling = analysis_df["fov"].apply(lambda x: fov_correction_dict[x]).persist()
        analysis_df[label + ": FOV Corrected"] = (analysis_df[label]/label_scaling).persist()
    else:
        analysis_df[label + ": FOV Corrected"] = analysis_df[label]
plt.savefig("FOV_correction.png",dpi=500)

### Variables over time

In [ ]:
analysis_df_nobkd = analysis_df[analysis_df["Objectid"]!=0]
values_to_rescale_step_2 = [value + ": FOV Corrected" for value in values_to_rescale]

In [ ]:
fig = plt.figure(figsize=(30,20))
values_names = ['Median mCherry Intensity', 'Median GFPmut2 Intensity']

for i,label in enumerate(values_to_rescale_step_2):
    time_series_groupby = analysis_df_nobkd.groupby("timepoints")[label]
    if label == 'mCherry-Penta mean_intensity: FOV Corrected':
        time_mode_series = time_series_groupby.apply(lambda x: bootstrap_hrm(x,max_n_per_bootstrap=100)).compute().sort_index()
    elif label == 'GFP-Penta mean_intensity: FOV Corrected':
        time_mode_series = get_GFPpos_modes(analysis_df['GFP-Penta mean_intensity'],time_series_groupby,max_n_per_bootstrap=100)
    else:
        print("Weird Label")
        
    time_correction_series = time_mode_series/np.max(time_mode_series)
    time_correction_dict = time_correction_series.to_dict()
    plt.subplot(2,3,i+1)
    plt.plot(time_correction_series)
    plt.title(values_names[i],fontsize=22)
    plt.xlabel("Timepoint (3 min steps)", fontsize=18)
    plt.ylabel("Scaling", fontsize=18)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18) 
    plt.ylim(0.,1.2)
    label_scaling = analysis_df["timepoints"].apply(lambda x: time_correction_dict[x])
    analysis_df[label + ": Time Corrected"] = (analysis_df[label]/label_scaling).persist()
plt.savefig("Time_correction.png",dpi=500)

### Overwrite Variables with Correction

In [ ]:
for label in values_to_rescale:
    analysis_df[label] = analysis_df[label + ': FOV Corrected: Time Corrected']

In [ ]:
analysis_df = analysis_df[['File Index', 'File Trench Index', 'timepoints', 'Objectid',
       'centroid_y', 'centroid_x', 'area', 'fov', 'row', 'trench', 'time (s)',
       'lane orientation', 'y (local)', 'x (local)', 'y (global)',
       'x (global)', 'trenchid', 'Trenchid Timepoint Index'] + values_to_rescale].persist()

### GFP Quantification Function

In [ ]:
def local_background_subtract(series,intensity_key):
    intensity_vals = series[intensity_key]
    bkd_val = series[series["Objectid"]==0][intensity_key].iloc[0]
    bkd_sub_intensity = intensity_vals-bkd_val
    bkd_sub_intensity = bkd_sub_intensity.to_dict()
    return bkd_sub_intensity

In [ ]:
analysis_df_trenchtimepoint_groupby = analysis_df.groupby("Trenchid Timepoint Index")

gfp_intensity_wo_bkd = analysis_df_trenchtimepoint_groupby.apply(lambda x: local_background_subtract(x,'GFP-Penta mean_intensity'),\
                                                        meta=('GFP-Penta mean_intensity',float)).compute().reset_index(drop=True).to_list()
gfp_intensity_wo_bkd = {k: v for d in gfp_intensity_wo_bkd for k, v in d.items()}
gfp_intensity_wo_bkd = pd.DataFrame.from_dict(gfp_intensity_wo_bkd,orient='index',columns=['GFP-Penta mean_intensity_wo_bkd'])

mchy_intensity_wo_bkd = analysis_df_trenchtimepoint_groupby.apply(lambda x: local_background_subtract(x,'mCherry-Penta mean_intensity'),\
                                                        meta=('mCherry-Penta mean_intensity',float)).compute().reset_index(drop=True).to_list()
mchy_intensity_wo_bkd = {k: v for d in mchy_intensity_wo_bkd for k, v in d.items()}
mchy_intensity_wo_bkd = pd.DataFrame.from_dict(mchy_intensity_wo_bkd,orient='index',columns=['mCherry-Penta mean_intensity_wo_bkd'])

In [ ]:
analysis_df = analysis_df.join(mchy_intensity_wo_bkd)
analysis_df = analysis_df.join(gfp_intensity_wo_bkd)
analysis_df_nobkd = analysis_df[analysis_df["Objectid"]!=0].persist()

ratio_series = analysis_df_nobkd['GFP-Penta mean_intensity_wo_bkd']/analysis_df_nobkd['mCherry-Penta mean_intensity_wo_bkd']
analysis_df_nobkd["gfp/mchy Ratio"] = ratio_series

#### Get Trench Mapping

In [ ]:
phenotype_kymopath = "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Growth_Division/kymograph/metadata"
barcode_kymopath = "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/kymograph/metadata"

trenchid_map = tr.files_to_trenchid_map(phenotype_kymopath,
                                        barcode_kymopath,
                                        offset_x=0)

# trenchid_map = tr.files_to_trenchid_map(phenotype_kymopath,
#                                         barcode_kymopath,
#                                         offset_x=-3)

# trenchid_map = tr.files_to_trenchid_map(phenotype_kymopath,
#                                         barcode_kymopath,
#                                         offset_x=0)

In [ ]:
# Load ./trenchid_map.pkl

import pickle as pkl
with open("./trenchid_map.pkl","rb") as f:
    trenchid_map = pkl.load(f)

In [ ]:
# Load kymo metadata for first one
# import dask.dataframe as dd
df_kymo = dd.read_parquet("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/kymograph/metadata",calculate_divisions=True)
df_bc = df_kymo.groupby('trenchid').first().compute()
df_bc
# df_single.to_csv('./df_single.csv')
# Load kymo metadata for second one

In [ ]:
df_bc[(df_bc['File Index'] == 2) & (df_bc['File Trench Index'] == 101)]

In [ ]:
# Load kymo metadata for first one
# import dask.dataframe as dd
df_kymo = dd.read_parquet("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Growth_Division/kymograph/metadata",calculate_divisions=True)
df_pheno = df_kymo.groupby('trenchid').first().compute()
df_pheno
# df_single.to_csv('./df_single.csv')
# Load kymo metadata for second one

In [ ]:
# Convert trenchid_map key, value pairs to array
# In bc df, in pheno df
trenchid_map_arr = np.array(list(trenchid_map.items()))
index = 500
print(trenchid_map_arr[index])
trenchid_bc, trenchid_ph = trenchid_map_arr[index][0],trenchid_map_arr[index][1]

print("BC trenchid: {}\nPhenotype trenchid: {}".format(trenchid_bc, trenchid_ph))
df_bc.loc[trenchid_bc:trenchid_bc+1]

bc_file_idx, bc_file_trench_idx = df_bc.loc[trenchid_bc]['File Index'], df_bc.loc[trenchid_bc]['File Trench Index']
ph_file_idx, ph_file_trench_idx = df_pheno.loc[trenchid_ph]['File Index'], df_pheno.loc[trenchid_ph]['File Trench Index']

print(bc_file_idx, bc_file_trench_idx)
print(ph_file_idx, ph_file_trench_idx)
# trenchid_map[

In [ ]:
### CHECK MAPPING
### QC segmentation
import h5py
import matplotlib.pyplot as plt
import numpy as np

channels = ["mCherry-Penta", "GFP-Penta"]
trench_index = ph_file_trench_idx  # BAD: 1 (k30), GOOD: k30-20
file_index = ph_file_idx
with h5py.File(
    "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Growth_Division/kymograph/kymograph_"
    + str(file_index)
    + ".hdf5",
    "r",
) as infile:
    indices = list(range(0, 2, 1))

    for ch in channels:
        img = infile[ch][trench_index][indices[0]]
        for ind in indices[1:]:
            img_ind = infile[ch][trench_index][ind]
            img = np.concatenate((img, img_ind), axis=1)
        plt.figure(figsize=(3, 6))
        plt.imshow(img, cmap="gray", aspect=1)
        plt.show()
        plt.close()

channels = ["data"]
# trench_index = 110  #BAD: 1 (k30), GOOD: k30-20
with h5py.File(
    "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Growth_Division/fluorsegmentation/segmentation_"
    + str(file_index)
    + ".hdf5",
    "r",
) as infile:
    indices = list(range(0, 2, 1))

    for ch in channels:
        img = infile[ch][trench_index][indices[0]]
        for ind in indices[1:]:
            img_ind = infile[ch][trench_index][ind]
            img = np.concatenate((img, img_ind), axis=1)
        plt.figure(figsize=(3, 6))
        plt.imshow(img, cmap="jet", aspect=1)
        plt.show()
        plt.close()

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

channels = ["BF", "RFP", "Cy5"]
file_index = bc_file_idx
trench_index = bc_file_trench_idx # BAD: 1 (k30), GOOD: k30-20
with h5py.File(
    "/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/Barcodes/kymograph/kymograph_"+
    str(file_index)+
    ".hdf5",
    "r",
) as infile:
    indices = list(range(0, 15, 1))
    for ch in channels:
        img = infile[ch][trench_index][indices[0]]
        for ind in indices[1:]:
            img_ind = infile[ch][trench_index][ind]
            img = np.concatenate((img, img_ind), axis=1)
        plt.figure(figsize=(4, 8))
        plt.imshow(img, cmap="gray", aspect=1)
        plt.show()
        plt.close()   

In [ ]:
pd.read_csv("/home/lag36/scratch/lag36/2024-02-12_lLAG2_Nanopore-Promethion_genome_Run_2/lLAG2_final_df.tsv", sep="\t")

In [ ]:
single_trench_timepoint_df[single_trench_timepoint_df['trenchid'] == 714].compute().loc[:,['Barcode', 'barcode']]

cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,2,3,8,9,14,15,20,21,26,27,17,22,23,28,29],
# barcode (lowercase): This is from sequencing
# Barcode (uppercase): This is from phenotype

In [ ]:
barcode = '000110000010111111110000010000'
Barcode = '000011110110101010011000010000'

In [ ]:
cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,2,3,8,9,14,15,20,21,26,27,17,22,23,28,29]

In [ ]:
''.join([barcode[ch_cyc] for ch_cyc in cycleorder])

In [ ]:
### CHECK MAPPING
### QC segmentation
import h5py
import matplotlib.pyplot as plt
import numpy as np

channels = ["mCherry-Penta", "GFP-Penta"]
trench_index = 999  # BAD: 1 (k30), GOOD: k30-20
file_index = 0
with h5py.File(
    "/home/lag36/scratch/lag36/2024-02-05_lLAG2_Run2_All-15-Cycles-AF555/Growth_Division/kymograph/kymograph_"
    + str(file_index)
    + ".hdf5",
    "r",
) as infile:
    indices = list(range(0, 2, 1))

    for ch in channels:
        img = infile[ch][trench_index][indices[0]]
        for ind in indices[1:]:
            img_ind = infile[ch][trench_index][ind]
            img = np.concatenate((img, img_ind), axis=1)
        plt.figure(figsize=(3, 6))
        plt.imshow(img, cmap="gray", aspect=1)
        plt.show()
        plt.close()

channels = ["data"]
# trench_index = 110  #BAD: 1 (k30), GOOD: k30-20
with h5py.File(
    "/home/lag36/scratch/lag36/2024-02-05_lLAG2_Run2_All-15-Cycles-AF555/Growth_Division/fluorsegmentation/segmentation_"
    + str(file_index)
    + ".hdf5",
    "r",
) as infile:
    indices = list(range(0, 2, 1))

    for ch in channels:
        img = infile[ch][trench_index][indices[0]]
        for ind in indices[1:]:
            img_ind = infile[ch][trench_index][ind]
            img = np.concatenate((img, img_ind), axis=1)
        plt.figure(figsize=(3, 6))
        plt.imshow(img, cmap="jet", aspect=1)
        plt.show()
        plt.close()

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

channels = ["BF", "RFP", "Cy5"]
file_index = 2
trench_index = 105 # BAD: 1 (k30), GOOD: k30-20
with h5py.File(
    "/home/lag36/scratch/lag36/2024-02-05_lLAG2_Run2_All-15-Cycles-AF555/Barcodes/kymograph/kymograph_"+
    str(file_index)+
    ".hdf5",
    "r",
) as infile:
    indices = list(range(0, 15, 1))
    for ch in channels:
        img = infile[ch][trench_index][indices[0]]
        for ind in indices[1:]:
            img_ind = infile[ch][trench_index][ind]
            img = np.concatenate((img, img_ind), axis=1)
        plt.figure(figsize=(4, 8))
        plt.imshow(img, cmap="gray", aspect=1)
        plt.show()
        plt.close()   

#### Get GFP Call Error and Recovery Rate

In [ ]:
import pickle 

with open('./trenchid_map.pkl', 'wb') as f:
    pickle.dump(trenchid_map, f)

In [ ]:
def get_barcode_pheno_df(phenotype_df, barcode_df, trenchid_map):
    ##phenotype_df must contain trenchids column and a File Parquet Index
    
    phenotype_df_idx = phenotype_df["trenchid"].unique().compute().tolist()
    valid_barcode_df = barcode_df[barcode_df["trenchid"].isin(trenchid_map.keys())].compute()
    barcode_df_mapped_trenchids = valid_barcode_df["trenchid"].apply(lambda x: trenchid_map[x])

    valid_init_df_indices = barcode_df_mapped_trenchids.isin(phenotype_df_idx)
    barcode_df_mapped_trenchids = barcode_df_mapped_trenchids[valid_init_df_indices]
    final_valid_barcode_df_indices = barcode_df_mapped_trenchids.index.to_list()

    called_df = barcode_df.loc[final_valid_barcode_df_indices]
    called_df["phenotype trenchid"] = barcode_df_mapped_trenchids
    called_df = called_df.reset_index().set_index("phenotype trenchid",drop=True,sorted=False).persist()

    output_df = phenotype_df.rename(columns={"trenchid":"phenotype trenchid"})
    output_df = output_df.reset_index().set_index("phenotype trenchid",drop=True,sorted=False)
    output_df = output_df.merge(called_df,how='inner',left_index=True,right_index=True)
    output_df = output_df.drop(["Barcode Signal"], axis=1)
    output_df = output_df.reset_index().set_index("File Parquet Index").persist()
    
    return output_df

In [ ]:
del df_kymo

In [ ]:
output_df = get_barcode_pheno_df(analysis_df_nobkd, barcode_df, trenchid_map)
# del analysis_df_nobkd

In [ ]:
print(1)

In [ ]:
output_df = output_df.repartition(npartitions=500)
output_df.to_parquet("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/2024-02-27_lLAG3_Analysis", engine="pyarrow", overwrite=True)

In [ ]:
output_df = dd.read_parquet("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/2024-02-27_lLAG3_Analysis",calculate_divisions=True)
output_df = output_df.set_index("Trenchid Timepoint Index",sorted=True).persist()
wait(output_df);
single_trench_timepoint_df = output_df.groupby("Trenchid Timepoint Index").apply(lambda x: x.iloc[0]).persist()
wait(single_trench_timepoint_df);

In [ ]:
single_trench_timepoint_df.to_parquet("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/2024-02-27_lLAG3_Analysis_Trench-Timepoint", engine="pyarrow", overwrite=True)

In [ ]:
single_trench_timepoint_df = single_trench_timepoint_df.compute()

In [ ]:
single_trench_timepoint_df.to_pickle("/home/lag36/scratch/lag36/2024-02-27_lLAG3_Run1/2024-02-27_lLAG3_Analysis_Trench-Timepoint_thresh_0.pkl")

In [ ]:
single_trench_timepoint_df.columns

In [ ]:

df_filtered = single_trench_timepoint_df.loc[(single_trench_timepoint_df['gfp/mchy Ratio'] > -.2) & (single_trench_timepoint_df['gfp/mchy Ratio'] < 1.5), ['gfp/mchy Ratio', 'gfp_is_bright']]

# single_trench_timepoint_df.loc[(single_trench_timepoint_df['gfp/mchy Ratio'] > -.2) & (single_trench_timepoint_df['gfp/mchy Ratio'] < 1.7), 'gfp/mchy Ratio'].compute().hist(bins=100)
# # plt.xlim(-0.25,2)

In [ ]:
df_filtered['gfp/mchy Ratio'].hist(bins=1000)
# plt.xlim([0.2,0.5])

In [ ]:
df_filtered['gfp/mchy Ratio'].hist(bins=1000)
# plt.xlim([0.2,0.5])

In [ ]:
cutoff_lo = 0.25
cutoff_hi = 0.35

In [ ]:
# Make a 3 row x 1 column grid of plots
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
# On the first row include the histogram of the gfp/mchy ratio
df_filtered['gfp/mchy Ratio'].hist(bins=100, ax=axs[0])
# On the second row include the histogram of the gfp/mchy ratio for the bright GFP
df_filtered[df_filtered['gfp_is_bright'] == 1]['gfp/mchy Ratio'].hist(bins=100, ax=axs[1])
# On the third row include the histogram of the gfp/mchy ratio for the dim GFP
df_filtered[df_filtered['gfp_is_bright'] == 0]['gfp/mchy Ratio'].hist(bins=100, ax=axs[2])
# On the fourth row, include the scatter plot
df_filtered.plot.scatter(x='gfp/mchy Ratio', y='gfp_is_bright', ax=axs[3], alpha=0.005)
# Draw the cutoff lines in all plots
for ax in axs:
    ax.axvline(cutoff_lo, color='r', linestyle='--')
    ax.axvline(cutoff_hi, color='r', linestyle='--')
# Annotate the top right of each plot
axs[0].annotate('All', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)
axs[1].annotate('Bright GFP', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)
axs[2].annotate('Dim GFP', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)
axs[3].annotate('Scatter', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)

# Remove y labels
# for ax in axs:
#     ax.set_ylabel('')


In [ ]:
pred_neg = df_filtered.loc[df_filtered['gfp/mchy Ratio'] < cutoff_lo, 'gfp_is_bright']
pred_pos = df_filtered.loc[df_filtered['gfp/mchy Ratio'] > cutoff_hi, 'gfp_is_bright']
print("{}/{} true / predicted negatives".format(pred_neg[pred_neg == 0].count(), pred_neg.count()))
print("Ratio: ", pred_neg[pred_neg == 0].count() / pred_neg.count())
print("{}/{} true / predicted positives".format(pred_pos[pred_pos == 1].count(), pred_pos.count()))
print("Ratio: ", pred_pos[pred_pos == 1].count() / pred_pos.count())

print("{}/{} total true calls / total calls".format(pred_neg[pred_neg == 0].count() + pred_pos[pred_pos == 1].count(), pred_neg.count() + pred_pos.count()))
print("Ratio: ", (pred_neg[pred_neg == 0].count() + pred_pos[pred_pos == 1].count()) / (pred_neg.count() + pred_pos.count()))

In [ ]:
# Make a 3 row x 1 column grid of plots
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
# On the first row include the histogram of the gfp/mchy ratio
df_filtered['gfp/mchy Ratio'].hist(bins=100, ax=axs[0])
# On the second row include the histogram of the gfp/mchy ratio for the bright GFP
df_filtered[df_filtered['gfp_is_bright'] == 1]['gfp/mchy Ratio'].hist(bins=100, ax=axs[1])
# On the third row include the histogram of the gfp/mchy ratio for the dim GFP
df_filtered[df_filtered['gfp_is_bright'] == 0]['gfp/mchy Ratio'].hist(bins=100, ax=axs[2])
# On the fourth row, include the scatter plot
df_filtered.plot.scatter(x='gfp/mchy Ratio', y='gfp_is_bright', ax=axs[3], alpha=0.005)
# Draw the cutoff lines in all plots
for ax in axs:
    ax.axvline(cutoff_lo, color='r', linestyle='--')
    ax.axvline(cutoff_hi, color='r', linestyle='--')
# Annotate the top right of each plot
axs[0].annotate('All', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)
axs[1].annotate('Bright GFP', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)
axs[2].annotate('Dim GFP', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)
axs[3].annotate('Scatter', xy=(0.95, 0.85), xycoords='axes fraction', ha='right', va='top', fontsize=20)

# Remove y labels
# for ax in axs:
#     ax.set_ylabel('')


In [ ]:
pred_neg = df_filtered.loc[df_filtered['gfp/mchy Ratio'] < cutoff_lo, 'gfp_is_bright']
pred_pos = df_filtered.loc[df_filtered['gfp/mchy Ratio'] > cutoff_hi, 'gfp_is_bright']

In [ ]:
print("{}/{} true / predicted negatives".format(pred_neg[pred_neg == 0].count(), pred_neg.count()))
print("Ratio: ", pred_neg[pred_neg == 0].count() / pred_neg.count())
print("{}/{} true / predicted positives".format(pred_pos[pred_pos == 1].count(), pred_pos.count()))
print("Ratio: ", pred_pos[pred_pos == 1].count() / pred_pos.count())

print("{}/{} total true calls / total calls".format(pred_neg[pred_neg == 0].count() + pred_pos[pred_pos == 1].count(), pred_neg.count() + pred_pos.count()))
print("Ratio: ", (pred_neg[pred_neg == 0].count() + pred_pos[pred_pos == 1].count()) / (pred_neg.count() + pred_pos.count()))

In [ ]:
dask_controller.shutdown()

In [ ]:
# Make a ROC plot
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, thresholds = roc_curve(df_filtered['gfp_is_bright'], df_filtered['gfp/mchy Ratio'])
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(fpr, tpr, label='ROC curve')
ax.plot([0, 1], [0, 1], linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curve')
ax.legend()
auc = roc_auc_score(df_filtered['gfp_is_bright'], df_filtered['gfp/mchy Ratio'])
print("AUC: ", auc)

# Make a precision-recall plot
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, thresholds = precision_recall_curve(df_filtered['gfp_is_bright'], df_filtered['gfp/mchy Ratio'])
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.plot(recall, precision, label='Precision-Recall curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curve')
ax.legend()
auc = average_precision_score(df_filtered['gfp_is_bright'], df_filtered['gfp/mchy Ratio'])
print("AUC: ", auc)



In [ ]:
single_trench_timepoint_df.columns

In [ ]:
dask_controller.shutdown()